# Phase 2 L-NER - Experiment 1: Full Fine-tuning (BIO)

**Local run — no Colab/Drive needed. Set paths in cell below.**

In [16]:
# Local run — no Colab/Drive needed
print("Running locally")

Running locally


## 1 - Imports & Setup

In [ ]:
import os, sys, re, json, math, time, random, bisect, copy, zipfile, glob, shutil
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

print("Imports OK")
print("PyTorch:", torch.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2 - Configuration

In [18]:
@dataclass
class NERConfig:
    # Model (must match Phase 1)
    vocab_size: int      = 40_000
    d_model: int         = 768
    n_layers: int        = 19
    n_query_heads: int   = 12
    n_kv_heads: int      = 4
    d_ff: int            = 2048
    max_seq_len: int     = 512
    dropout: float       = 0.1
    weight_tying: bool   = True
    rope_theta: float    = 10000.0

    # NER
    num_labels: int      = 12
    bio_num_tags: int    = 25      # BIO: 2 * num_labels + 1 (O)
    window_size: int     = 510
    window_stride: int   = 255

    # Training
    lr: float            = 5e-6
    min_lr: float        = 1e-7
    warmup_steps: int    = 200
    num_epochs: int      = 25
    batch_size: int      = 8       # minimal batch for VRAM
    grad_accum: int      = 2       # effective batch = 2*8 = 16
    weight_decay: float  = 0.01
    max_grad_norm: float = 1.0
    use_amp: bool        = True
    seed: int            = 42

    def __post_init__(self):
        self.head_dim = self.d_model // self.n_query_heads
        self.n_groups = self.n_query_heads // self.n_kv_heads

CFG = NERConfig()

OUT       = Path(".")
CKPT_DIR  = OUT / "ner_checkpoints"; CKPT_DIR.mkdir(exist_ok=True)
SUB_DIR   = OUT / "submissions";     SUB_DIR.mkdir(exist_ok=True)

print(f"Config: {CFG.d_model}d x {CFG.n_layers}L, window={CFG.window_size}, stride={CFG.window_stride}")
print(f"Batch={CFG.batch_size} x grad_accum={CFG.grad_accum} = effective {CFG.batch_size * CFG.grad_accum}")

Config: 768d x 19L, window=510, stride=255
Batch=8 x grad_accum=2 = effective 16


## 3 - Find Files & Load Tokenizer

In [19]:
# ── Point these to your local files ──
PHASE1_CKPT   = "best.pt"           # Phase-1 checkpoint
tok_json_path = "bpe_40k.json"      # BPE vocab
tok_py_path   = "a1_tokenizer.py"   # tokenizer module

for f in [PHASE1_CKPT, tok_json_path, tok_py_path]:
    assert os.path.isfile(f), f"File not found: {f}  — put it in the working directory"

sys.path.insert(0, ".")
from a1_tokenizer import BPETokenizer
tokenizer = BPETokenizer.load(tok_json_path)
CFG.vocab_size = tokenizer.vocab_size
print(f"Tokenizer loaded: vocab_size={tokenizer.vocab_size}")

_t = "The Supreme Court held that"
assert tokenizer.decode(tokenizer.encode(_t)) == _t, "Round-trip failed"
print("Tokenizer OK")

Tokenizer loaded: vocab_size=40000
Tokenizer OK


## 4 - Download NER Data

In [ ]:
!pip -q install huggingface_hub
from huggingface_hub import hf_hub_download

HF_REPO = "Exploration-Lab/CS-781-Capstone"
HF_TOKEN = ""
print("HF token set")

NER_FILES = {
    "fold1_text":  "Phase_2_LNER/Train/fold1-text.json",
    "fold1_gold":  "Phase_2_LNER/Train/fold1-gold.json",
    "fold2_text":  "Phase_2_LNER/Train/fold2-text.json",
    "fold2_gold":  "Phase_2_LNER/Train/fold2-gold.json",
    "labels":      "Phase_2_LNER/Train/labels.txt",
    "fold3_text":  "Phase_2_LNER/Test/fold3-text.json",
}

ner_paths = {}
for key, fn in NER_FILES.items():
    ner_paths[key] = hf_hub_download(HF_REPO, fn, repo_type="dataset", token=HF_TOKEN)
    print(f"  {key:12s} -> downloaded")

HF token set
  fold1_text   -> downloaded
  fold1_gold   -> downloaded
  fold2_text   -> downloaded
  fold2_gold   -> downloaded
  labels       -> downloaded
  fold3_text   -> downloaded


## 5 - Parse Data & Label Maps

In [21]:
with open(ner_paths["labels"], "r") as f:
    LABELS = [l.strip() for l in f if l.strip()]
print(f"Entity labels ({len(LABELS)}): {LABELS}")

# BIO tag scheme: B-X and I-X for each entity type, plus O
O_TAG = 0
tag2id = {"O": O_TAG}
id2tag = {O_TAG: "O"}
for i, lab in enumerate(LABELS):
    tag2id[f"B-{lab}"] = 2*i + 1
    tag2id[f"I-{lab}"] = 2*i + 2
    id2tag[2*i + 1] = f"B-{lab}"
    id2tag[2*i + 2] = f"I-{lab}"

CFG.num_labels = len(LABELS)
CFG.bio_num_tags = len(tag2id)   # 2*num_labels + 1
print(f"BIO tags: {CFG.bio_num_tags} ({len(LABELS)} labels * 2 + O)")

def load_json(p):
    with open(p, "r") as f:
        return json.load(f)

fold1_text = load_json(ner_paths["fold1_text"])
fold1_gold = load_json(ner_paths["fold1_gold"])
fold2_text = load_json(ner_paths["fold2_text"])
fold2_gold = load_json(ner_paths["fold2_gold"])
fold3_text = load_json(ner_paths["fold3_text"])

print(f"Fold 1: {len(fold1_text)} docs")
print(f"Fold 2: {len(fold2_text)} docs")
print(f"Fold 3 (test): {len(fold3_text)} docs")

Entity labels (12): ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
BIO tags: 25 (12 labels * 2 + O)
Fold 1: 35 docs
Fold 2: 35 docs
Fold 3 (test): 35 docs


## 6 - Offset-Tracking Encoder

In [22]:
def encode_with_offsets(tokenizer, text):
    token_ids = []
    char_spans = []
    for m in re.finditer(r'\S+', text):
        word = m.group()
        ws = m.start()
        char_byte_bounds = [0]
        for ch in word:
            char_byte_bounds.append(char_byte_bounds[-1] + len(ch.encode('utf-8')))
        bpe_syms = tokenizer._bpe(word)
        SPACE_BYTES = 1
        fb_pos = 0
        for sym in bpe_syms:
            tok_id = tokenizer.token_to_id.get(sym, tokenizer.token_to_id.get('<unk>', 1))
            n_bytes = len(sym)
            fb_end = fb_pos + n_bytes
            wb_s = max(0, fb_pos - SPACE_BYTES)
            wb_e = max(0, fb_end - SPACE_BYTES)
            wb_e = min(wb_e, char_byte_bounds[-1])
            wb_s = min(wb_s, char_byte_bounds[-1])
            if wb_e > wb_s:
                c_s = bisect.bisect_right(char_byte_bounds, wb_s) - 1
                c_e = bisect.bisect_left(char_byte_bounds, wb_e)
                c_s = max(0, c_s)
                c_e = min(len(word), c_e)
                char_spans.append((ws + c_s, ws + c_e))
            else:
                char_spans.append((ws, ws))
            token_ids.append(tok_id)
            fb_pos = fb_end
    return token_ids, char_spans

def get_doc_text(doc_entry):
    if isinstance(doc_entry, str):
        return doc_entry
    if isinstance(doc_entry, dict):
        return doc_entry.get("text", "")
    return str(doc_entry)

print("Offset encoder defined")

Offset encoder defined


## 7 - Model Architecture

In [23]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).to(x.dtype) * self.weight

def precompute_rope(dim, max_len, theta=10000.0, device=None):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))
    t = torch.arange(max_len, device=device).float()
    angles = torch.outer(t, freqs)
    return angles.cos(), angles.sin()

def apply_rope(x, cos, sin, offset=0):
    T = x.size(2)
    c = cos[offset:offset+T].unsqueeze(0).unsqueeze(0)
    s = sin[offset:offset+T].unsqueeze(0).unsqueeze(0)
    hd = x.size(-1) // 2
    x1, x2 = x[..., :hd], x[..., hd:]
    rotated = torch.cat([-x2, x1], dim=-1)
    c_exp = torch.cat([c.expand(-1,-1,-1,hd), c.expand(-1,-1,-1,hd)], dim=-1)
    s_exp = torch.cat([s.expand(-1,-1,-1,hd), s.expand(-1,-1,-1,hd)], dim=-1)
    return x * c_exp + rotated * s_exp

class GQA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nqh = cfg.n_query_heads
        self.nkh = cfg.n_kv_heads
        self.ng  = cfg.n_groups
        self.hd  = cfg.head_dim
        self.dm  = cfg.d_model
        self.q = nn.Linear(cfg.d_model, cfg.n_query_heads * cfg.head_dim, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.n_kv_heads   * cfg.head_dim, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.n_kv_heads   * cfg.head_dim, bias=False)
        self.o = nn.Linear(cfg.n_query_heads * cfg.head_dim, cfg.d_model, bias=False)
        self.attn_drop  = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.scale = 1.0 / math.sqrt(cfg.head_dim)

    def forward(self, x, cos, sin, mask=None, kv_cache=None, offset=0):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nqh, self.hd).transpose(1,2)
        k = self.k(x).view(B, T, self.nkh, self.hd).transpose(1,2)
        v = self.v(x).view(B, T, self.nkh, self.hd).transpose(1,2)
        q = apply_rope(q, cos, sin, offset)
        k = apply_rope(k, cos, sin, offset)
        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = (k, v)
        if self.ng > 1:
            k = k.unsqueeze(2).expand(B, self.nkh, self.ng, k.size(2), self.hd).reshape(B, self.nqh, -1, self.hd)
            v = v.unsqueeze(2).expand(B, self.nkh, self.ng, v.size(2), self.hd).reshape(B, self.nqh, -1, self.hd)
        w = torch.matmul(q, k.transpose(-2,-1)) * self.scale
        if mask is not None: w = w + mask
        w = F.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)
        w = self.attn_drop(w)
        out = torch.matmul(w, v).transpose(1,2).contiguous().view(B, T, self.dm)
        return self.resid_drop(self.o(out)), new_cache

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up   = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn  = GQA(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn   = SwiGLU(cfg)
    def forward(self, x, cos, sin, mask=None, kv_cache=None, offset=0):
        h, cache = self.attn(self.norm1(x), cos, sin, mask, kv_cache, offset)
        x = x + h
        x = x + self.ffn(self.norm2(x))
        return x, cache

class LegalLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb  = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop     = nn.Dropout(cfg.dropout)
        self.layers   = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm     = RMSNorm(cfg.d_model)
        self.lm_head  = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.weight_tying:
            self.lm_head.weight = self.tok_emb.weight
        cos, sin = precompute_rope(cfg.head_dim, cfg.max_seq_len * 2, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def _causal_mask(self, T, device, S=None):
        if S is None: S = T
        m = torch.full((T, S), float("-inf"), device=device)
        off = S - T
        for i in range(T):
            m[i, :off+i+1] = 0.0
        return m.unsqueeze(0).unsqueeze(0)

    def forward(self, ids, targets=None, kv_caches=None, offset=0):
        B, T = ids.shape
        x = self.drop(self.tok_emb(ids))
        mask = self._causal_mask(T, ids.device, T + offset)
        new_caches = []
        for i, layer in enumerate(self.layers):
            c = kv_caches[i] if kv_caches else None
            x, nc = layer(x, self.rope_cos, self.rope_sin, mask, c, offset)
            new_caches.append(nc)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.cfg.vocab_size),
                                   targets.view(-1), ignore_index=-1)
        return logits, loss, new_caches

print("Model architecture defined")

Model architecture defined


## 8 - Load Phase 1 Backbone

In [24]:
def load_phase1_backbone(cfg, ckpt_path, device):
    model = LegalLM(cfg)
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state = ckpt.get("model", ckpt)
    model.load_state_dict(state, strict=True)
    model = model.to(device)
    print(f"Phase 1 backbone loaded from {os.path.basename(ckpt_path)}")
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
    return model

print("Backbone loader defined")

Backbone loader defined


## 9 - NER Model (BiLSTM-CRF with CRF forward alg FIXED)

In [25]:
# ========================================================
# CRITICAL FIX: CRF forward algorithm emission dimension
# OLD BUG: emit = emissions[:, t].unsqueeze(1)  -> wrong dim
# NEW FIX: emit = emissions[:, t].unsqueeze(2)  -> correct dim
# ========================================================

from torch.utils.checkpoint import checkpoint as ckpt_fn

class CRF(nn.Module):
    """Linear-chain CRF with BUG-FIXED forward algorithm."""
    def __init__(self, num_tags):
        super().__init__()
        self.num_tags = num_tags
        self.transitions = nn.Parameter(torch.randn(num_tags, num_tags))
        self.start_transitions = nn.Parameter(torch.randn(num_tags))
        self.end_transitions = nn.Parameter(torch.randn(num_tags))
        nn.init.uniform_(self.transitions, -0.1, 0.1)
        nn.init.uniform_(self.start_transitions, -0.1, 0.1)
        nn.init.uniform_(self.end_transitions, -0.1, 0.1)

    def forward(self, emissions, tags, mask):
        gold_score = self._score_sentence(emissions, tags, mask)
        forward_score = self._forward_alg(emissions, mask)
        return (forward_score - gold_score).mean()

    def _score_sentence(self, emissions, tags, mask):
        B, T, C = emissions.shape
        score = self.start_transitions[tags[:, 0]] + emissions[:, 0].gather(1, tags[:, 0:1]).squeeze(1)
        for t in range(1, T):
            trans = self.transitions[tags[:, t], tags[:, t-1]]
            emit = emissions[:, t].gather(1, tags[:, t:t+1]).squeeze(1)
            step_score = trans + emit
            score = torch.where(mask[:, t], score + step_score, score)
        last_idx = mask.long().sum(dim=1) - 1
        last_tags = tags.gather(1, last_idx.unsqueeze(1)).squeeze(1)
        score = score + self.end_transitions[last_tags]
        return score

    def _forward_alg(self, emissions, mask):
        B, T, C = emissions.shape
        alpha = self.start_transitions.unsqueeze(0) + emissions[:, 0]
        for t in range(1, T):
            # ============ BUG FIX ============
            emit = emissions[:, t].unsqueeze(2)   # FIXED: was unsqueeze(1)
            # =================================
            trans = self.transitions.unsqueeze(0)
            alpha_exp = alpha.unsqueeze(1) + trans + emit
            new_alpha = torch.logsumexp(alpha_exp, dim=2)
            alpha = torch.where(mask[:, t].unsqueeze(1), new_alpha, alpha)
        alpha = alpha + self.end_transitions.unsqueeze(0)
        return torch.logsumexp(alpha, dim=1)

    def decode(self, emissions, mask):
        B, T, C = emissions.shape
        viterbi = emissions[:, 0] + self.start_transitions.unsqueeze(0)
        backpointers = []
        for t in range(1, T):
            trans = self.transitions.unsqueeze(0)
            scores = viterbi.unsqueeze(1) + trans
            best_scores, best_ids = scores.max(dim=2)
            emit = emissions[:, t]
            new_viterbi = best_scores + emit
            viterbi = torch.where(mask[:, t].unsqueeze(1), new_viterbi, viterbi)
            backpointers.append(best_ids)
        viterbi = viterbi + self.end_transitions.unsqueeze(0)
        lengths = mask.long().sum(dim=1)
        best_paths = []
        for b in range(B):
            seq_len = lengths[b].item()
            best_last = viterbi[b].argmax().item()
            path = [best_last]
            for t in range(seq_len - 2, -1, -1):
                best_last = backpointers[t][b, best_last].item()
                path.append(best_last)
            path.reverse()
            path = path + [0] * (T - seq_len)
            best_paths.append(path)
        return best_paths


class LegalNER(nn.Module):
    def __init__(self, backbone, num_tags, dropout=0.3, freeze_layers=10):
        super().__init__()
        self.backbone = backbone
        self.num_tags = num_tags
        d_model = backbone.tok_emb.weight.shape[1]
        self.d_model = d_model
        self.freeze_layers = freeze_layers
        self.bilstm = nn.LSTM(
            input_size=d_model,
            hidden_size=d_model // 2,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout,
        )
        self.ner_dropout = nn.Dropout(dropout)
        self.ner_head = nn.Linear(d_model, num_tags)
        self.crf = CRF(num_tags)
        nn.init.normal_(self.ner_head.weight, std=0.02)
        nn.init.zeros_(self.ner_head.bias)

        # Freeze first N backbone layers — saves optimizer memory + faster
        for i, layer in enumerate(self.backbone.layers):
            if i < freeze_layers:
                for p in layer.parameters():
                    p.requires_grad = False
        # Freeze embeddings too
        self.backbone.tok_emb.weight.requires_grad = False
        n_frozen = sum(1 for p in self.backbone.parameters() if not p.requires_grad)
        n_total = sum(1 for p in self.backbone.parameters())
        print(f"Frozen {n_frozen}/{n_total} backbone param tensors (first {freeze_layers}/{len(self.backbone.layers)} layers + embeddings)")

    def get_hidden_states(self, ids, attention_mask=None):
        B, T = ids.shape
        x = self.backbone.drop(self.backbone.tok_emb(ids))
        causal = self.backbone._causal_mask(T, ids.device)
        if attention_mask is not None:
            pad_mask = torch.zeros(B, 1, 1, T, device=ids.device, dtype=x.dtype)
            pad_mask.masked_fill_(attention_mask[:, None, None, :] == 0, float("-inf"))
            mask = causal + pad_mask
        else:
            mask = causal
        for i, layer in enumerate(self.backbone.layers):
            if i < self.freeze_layers:
                # Frozen layers — no grad, no checkpointing needed
                with torch.no_grad():
                    x, _ = layer(x, self.backbone.rope_cos, self.backbone.rope_sin, mask)
                x = x.detach()  # cut the graph here
            else:
                # Trainable layers — use gradient checkpointing to save VRAM
                def create_custom_forward(module):
                    def custom_forward(x, cos, sin, mask):
                        out, _ = module(x, cos, sin, mask)
                        return out
                    return custom_forward
                x = ckpt_fn(create_custom_forward(layer),
                           x, self.backbone.rope_cos, self.backbone.rope_sin, mask,
                           use_reentrant=False)
        x = self.backbone.norm(x)
        return x

    def forward(self, input_ids, labels=None, attention_mask=None):
        hidden = self.get_hidden_states(input_ids, attention_mask=attention_mask)
        hidden, _ = self.bilstm(hidden)
        emissions = self.ner_head(self.ner_dropout(hidden))
        out = {"logits": emissions}
        if labels is not None:
            crf_mask = (labels != -100)
            safe_labels = labels.clone()
            safe_labels[~crf_mask] = 0
            out["loss"] = self.crf(emissions, safe_labels, crf_mask)
        return out

    def decode(self, input_ids, attention_mask=None):
        hidden = self.get_hidden_states(input_ids, attention_mask=attention_mask)
        hidden, _ = self.bilstm(hidden)
        emissions = self.ner_head(hidden)
        crf_mask = attention_mask.bool() if attention_mask is not None else torch.ones(input_ids.shape, dtype=torch.bool, device=input_ids.device)
        return self.crf.decode(emissions, crf_mask)

print(f"LegalNER + BiLSTM-CRF: freeze first 10/19 layers + grad checkpointing on rest")

LegalNER + BiLSTM-CRF: freeze first 10/19 layers + grad checkpointing on rest


## 10 - NER Dataset (Sliding Windows)

In [26]:
def assign_bio_tags(char_spans, gold_annotations, tag2id):
    """BIO tag assignment: B- for first token of entity, I- for continuation."""
    n_tokens = len(char_spans)
    tags = [O_TAG] * n_tokens
    golds = sorted(gold_annotations, key=lambda a: int(a[0]))
    for ann in golds:
        gs, ge, label = int(ann[0]), int(ann[1]), ann[2]
        if label not in LABELS:
            continue
        b_tag = tag2id.get(f"B-{label}", O_TAG)
        i_tag = tag2id.get(f"I-{label}", O_TAG)
        first = True
        for t_idx, (cs, ce) in enumerate(char_spans):
            if ce <= cs:
                continue
            if cs < ge and ce > gs:
                tags[t_idx] = b_tag if first else i_tag
                first = False
    return tags

class NERDataset(Dataset):
    def __init__(self, text_dict, gold_dict, tokenizer, cfg, tag2id):
        self.windows = []
        self.cfg = cfg
        for doc_id in tqdm(text_dict, desc="Building NER dataset"):
            text = get_doc_text(text_dict[doc_id])
            if not text.strip():
                continue
            all_ids, all_spans = encode_with_offsets(tokenizer, text)
            n = len(all_ids)
            gold_anns = gold_dict.get(doc_id, [])
            all_tags = assign_bio_tags(all_spans, gold_anns, tag2id)
            if n == 0:
                continue
            step = cfg.window_stride if cfg.window_stride > 0 else cfg.window_size
            starts = list(range(0, max(1, n - cfg.window_size + 1), step))
            if starts[-1] + cfg.window_size < n:
                starts.append(n - cfg.window_size)
            for w_idx, s in enumerate(starts):
                e = min(s + cfg.window_size, n)
                w_ids = all_ids[s:e]
                w_tags = all_tags[s:e]
                self.windows.append((w_ids, w_tags, doc_id, w_idx))
        print(f"  {len(self.windows)} windows from {len(text_dict)} docs")

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w_ids, w_tags, doc_id, w_idx = self.windows[idx]
        L = self.cfg.window_size
        pad_len = L - len(w_ids)
        ids = w_ids + [0] * pad_len
        tags = w_tags + [-100] * pad_len
        mask = [1] * len(w_ids) + [0] * pad_len
        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "labels":    torch.tensor(tags, dtype=torch.long),
            "attention_mask": torch.tensor(mask, dtype=torch.long),
        }

print("NERDataset defined")

NERDataset defined


In [27]:
all_text = {**fold1_text, **fold2_text}
all_gold = {**fold1_gold, **fold2_gold}
doc_ids = sorted(all_text.keys())
random.seed(CFG.seed)
random.shuffle(doc_ids)
split_idx = len(doc_ids) - 3   # 3 val docs instead of 7
train_ids = doc_ids[:split_idx]
val_ids = doc_ids[split_idx:]
train_text = {d: all_text[d] for d in train_ids}
train_gold = {d: all_gold[d] for d in train_ids}
val_text = {d: all_text[d] for d in val_ids}
val_gold = {d: all_gold[d] for d in val_ids}
print(f"Train: {len(train_text)} docs, Val: {len(val_text)} docs")

print("Building train dataset ...")
train_ds = NERDataset(train_text, train_gold, tokenizer, CFG, tag2id)
print("Building val dataset ...")
val_ds   = NERDataset(val_text, val_gold, tokenizer, CFG, tag2id)

# Oversample windows containing rare entities
from collections import Counter
rare_label_names = ["WIT", "RESP", "APP", "R.COUNSEL", "A.COUNSEL", "JUDGE"]
rare_tags = set()
for ln in rare_label_names:
    for prefix in ["B-", "I-"]:
        if (prefix + ln) in tag2id:
            rare_tags.add(tag2id[prefix + ln])
wit_tags = set()
for prefix in ["B-", "I-"]:
    if (prefix + "WIT") in tag2id:
        wit_tags.add(tag2id[prefix + "WIT"])
orig_count = len(train_ds.windows)
oversampled = []
for w_ids, w_tags, doc_id, w_idx in train_ds.windows:
    oversampled.append((w_ids, w_tags, doc_id, w_idx))
    tag_set = set(w_tags)
    if tag_set & wit_tags:
        for _ in range(20):  # 20x for WIT
            oversampled.append((w_ids, w_tags, doc_id, w_idx))
    else:
        n_rare = len(tag_set & rare_tags)
        if n_rare > 0:
            for _ in range(5 * n_rare):
                oversampled.append((w_ids, w_tags, doc_id, w_idx))
train_ds.windows = oversampled
print(f"Oversampling: {orig_count} -> {len(train_ds.windows)} windows")

train_dl = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                      num_workers=4, pin_memory=True, drop_last=True)
val_dl   = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False,
                      num_workers=4, pin_memory=True)

print(f"Train: {len(train_ds)} windows, Val: {len(val_ds)} windows")

Train: 67 docs, Val: 3 docs
Building train dataset ...


Building NER dataset:   0%|          | 0/67 [00:00<?, ?it/s]

  1531 windows from 67 docs
Building val dataset ...


Building NER dataset:   0%|          | 0/3 [00:00<?, ?it/s]

  61 windows from 3 docs
Oversampling: 1531 -> 12166 windows
Train: 12166 windows, Val: 61 windows


## 11 - Training & Doc-Level Evaluation (matches leaderboard)

In [28]:
# ============================================================
# DOC-LEVEL VALIDATION: mirrors leaderboard evaluation.
# For each val doc:
#   1. Encode with BPE + char offsets
#   2. Sliding windows -> cache best-logit per token (center-pref)
#   3. CRF Viterbi over full doc emissions
#   4. Convert token tags -> char-level spans (with trim)
#   5. Compare against ORIGINAL gold char annotations
# No per-window double-counting; no token/char mismatch.
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_cosine_mult(step, total_steps, warmup_steps):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * prog))

TRIM_CHARS = set('()[]{},:;/\\\'\"---!?.')

def _trim_spans(doc_spans, text):
    """Strip leading/trailing punctuation from char spans."""
    trimmed = []
    for s_start, s_end, s_label in doc_spans:
        if s_start >= s_end:
            continue
        snippet = text[s_start:s_end]
        l = 0
        while l < len(snippet) and snippet[l] in TRIM_CHARS:
            l += 1
        r = len(snippet)
        while r > l and snippet[r - 1] in TRIM_CHARS:
            r -= 1
        if r > l:
            trimmed.append([s_start + l, s_start + r, s_label])
    return trimmed

def tag_preds_to_char_spans(tag_preds, char_spans, text):
    """BIO: convert tag IDs to char-level spans. B- starts, I-X continues iff same type."""
    doc_spans = []
    current_label = None
    span_start_char = None
    span_end_char = None
    for i, tag_id in enumerate(tag_preds):
        tag_name = id2tag.get(tag_id, "O")
        cs, ce = char_spans[i]
        if tag_name.startswith("B-"):
            if current_label is not None and span_start_char is not None:
                doc_spans.append([span_start_char, span_end_char, current_label])
            current_label = tag_name[2:]
            span_start_char = cs
            span_end_char = ce
        elif tag_name.startswith("I-"):
            if current_label == tag_name[2:]:
                span_end_char = max(span_end_char, ce) if span_end_char else ce
            else:
                if current_label is not None and span_start_char is not None:
                    doc_spans.append([span_start_char, span_end_char, current_label])
                current_label = tag_name[2:]
                span_start_char = cs
                span_end_char = ce
        else:  # O
            if current_label is not None and span_start_char is not None:
                doc_spans.append([span_start_char, span_end_char, current_label])
            current_label = None
            span_start_char = None
            span_end_char = None
    if current_label is not None and span_start_char is not None:
        doc_spans.append([span_start_char, span_end_char, current_label])
    return _trim_spans(doc_spans, text)

@torch.no_grad()
def evaluate_ner_doc_level(model, text_dict, gold_dict, tokenizer, cfg, device=DEVICE):
    """Doc-level evaluation at char-span level — matches leaderboard scoring."""
    model.eval()
    pred_spans_all = defaultdict(set)
    gold_spans_all = defaultdict(set)
    total_loss = 0.0
    n_batches = 0

    for doc_id in text_dict:
        text = get_doc_text(text_dict[doc_id])
        if not text.strip():
            continue
        all_ids, all_char_spans = encode_with_offsets(tokenizer, text)
        n = len(all_ids)
        if n == 0:
            continue

        gold_anns = gold_dict.get(doc_id, [])
        full_tags = assign_bio_tags(all_char_spans, gold_anns, tag2id)

        step = cfg.window_stride if cfg.window_stride > 0 else cfg.window_size
        starts = list(range(0, max(1, n - cfg.window_size + 1), step))
        if starts[-1] + cfg.window_size < n:
            starts.append(max(0, n - cfg.window_size))

        token_logits = [None] * n
        token_dist   = [-1] * n
        for s in starts:
            e = min(s + cfg.window_size, n)
            w_ids  = all_ids[s:e]
            w_tags = full_tags[s:e]
            pad_len = cfg.window_size - len(w_ids)
            inp   = torch.tensor([w_ids + [0]*pad_len], dtype=torch.long, device=device)
            labs  = torch.tensor([w_tags + [-100]*pad_len], dtype=torch.long, device=device)
            amask = torch.tensor([[1]*len(w_ids) + [0]*pad_len], dtype=torch.long, device=device)
            with torch.amp.autocast('cuda', enabled=cfg.use_amp):
                out = model(inp, labels=labs, attention_mask=amask)
            total_loss += out["loss"].item()
            n_batches += 1
            logits = out["logits"][0, :len(w_ids), :]
            for i_in_w in range(len(w_ids)):
                g = s + i_in_w
                dist = min(i_in_w, len(w_ids) - 1 - i_in_w)
                if dist > token_dist[g]:
                    token_dist[g] = dist
                    token_logits[g] = logits[i_in_w]

        # Global CRF Viterbi on best-cached emissions
        emissions_list = []
        for i in range(n):
            if token_logits[i] is not None:
                emissions_list.append(token_logits[i])
            else:
                emissions_list.append(torch.zeros(model.num_tags, device=device))
        emissions = torch.stack(emissions_list).unsqueeze(0)
        crf_mask  = torch.ones(1, n, dtype=torch.bool, device=device)
        tag_preds = model.crf.decode(emissions, crf_mask)[0]

        # Char-level predicted spans (trimmed)
        pred_char = tag_preds_to_char_spans(tag_preds, all_char_spans, text)
        for ps, pe, pl in pred_char:
            pred_spans_all[pl].add((doc_id, ps, pe))

        # Gold char spans direct from original annotations
        for ann in gold_anns:
            gs, ge, label = int(ann[0]), int(ann[1]), ann[2]
            if label in LABELS:
                gold_spans_all[label].add((doc_id, gs, ge))

    per_label = {}
    for label in LABELS:
        ps = pred_spans_all.get(label, set())
        gs = gold_spans_all.get(label, set())
        tp = len(ps & gs)
        fp = len(ps - gs)
        fn = len(gs - ps)
        p = tp / (tp + fp) if tp + fp > 0 else 0
        r = tp / (tp + fn) if tp + fn > 0 else 0
        f1 = 2 * p * r / (p + r) if p + r > 0 else 0
        per_label[label] = f1
    macro_f1 = sum(per_label.values()) / len(LABELS) if LABELS else 0
    avg_loss = total_loss / max(n_batches, 1)
    return avg_loss, macro_f1, per_label


def train_ner(model, train_dl, val_text, val_gold, cfg, exp_name, device=DEVICE, patience=5):
    set_seed(cfg.seed)
    model = model.to(device)

    backbone_params = [(n, p) for n, p in model.named_parameters()
                       if "backbone" in n and p.requires_grad]
    head_params = [(n, p) for n, p in model.named_parameters()
                   if "backbone" not in n and p.requires_grad]
    head_lr = getattr(cfg, 'head_lr', 1e-3)

    param_groups = []
    if backbone_params:
        param_groups.append({"params": [p for _, p in backbone_params], "lr": cfg.lr})
    if head_params:
        param_groups.append({"params": [p for _, p in head_params], "lr": head_lr})

    n_backbone = sum(p.numel() for _, p in backbone_params)
    n_head = sum(p.numel() for _, p in head_params)
    print(f"Backbone: {n_backbone:,} (lr={cfg.lr:.1e}) | Head: {n_head:,} (lr={head_lr:.1e})")

    optimizer = torch.optim.AdamW(param_groups, weight_decay=cfg.weight_decay)
    base_lrs = [pg["lr"] for pg in optimizer.param_groups]
    scaler = torch.amp.GradScaler('cuda', enabled=cfg.use_amp)
    total_steps = len(train_dl) * cfg.num_epochs // cfg.grad_accum
    best_f1 = 0.0
    best_state = None
    global_step = 0
    no_improve = 0
    all_trainable = [p for p in model.parameters() if p.requires_grad]

    for epoch in range(cfg.num_epochs):
        model.train()
        running_loss = 0.0
        t0 = time.time()
        for batch_idx, batch in enumerate(train_dl):
            ids  = batch["input_ids"].to(device)
            labs = batch["labels"].to(device)
            amask = batch["attention_mask"].to(device)
            with torch.amp.autocast('cuda', enabled=cfg.use_amp):
                out = model(ids, labels=labs, attention_mask=amask)
                loss = out["loss"] / cfg.grad_accum
            scaler.scale(loss).backward()
            running_loss += loss.item()
            if (batch_idx + 1) % cfg.grad_accum == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(all_trainable, cfg.max_grad_norm)
                mult = get_cosine_mult(global_step, total_steps, cfg.warmup_steps)
                for pg, blr in zip(optimizer.param_groups, base_lrs):
                    pg["lr"] = max(cfg.min_lr, blr * mult)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1
            if (batch_idx + 1) % (50 * cfg.grad_accum) == 0:
                avg = running_loss / (batch_idx + 1)
                elapsed = time.time() - t0
                print(f"  [{exp_name}] Ep {epoch+1} step {batch_idx+1}/{len(train_dl)} | loss {avg:.4f} | {elapsed:.0f}s")
        val_loss, macro_f1, per_label = evaluate_ner_doc_level(
            model, val_text, val_gold, tokenizer, cfg, device)
        print(f"  [{exp_name}] Epoch {epoch+1} | val_loss={val_loss:.4f} | DOC macro_f1={macro_f1:.4f}")
        for lab, f1 in sorted(per_label.items(), key=lambda x: -x[1]):
            print(f"    {lab:30s} F1={f1:.4f}")
        if macro_f1 > best_f1:
            best_f1 = macro_f1
            best_state = copy.deepcopy(model.state_dict())
            ckpt_path = CKPT_DIR / f"{exp_name}_best.pt"
            torch.save({"model": best_state, "f1": best_f1, "epoch": epoch+1}, ckpt_path)
            print(f"  New best DOC F1={best_f1:.4f}")
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"  Training done. Best DOC macro-F1: {best_f1:.4f}")
    return model

print("Training + doc-level evaluation defined")

Training + doc-level evaluation defined


## 12 - Prediction & Submission (CRF global decode + trimming)

In [29]:
@torch.no_grad()
def predict_test(model, text_dict, tokenizer, cfg, device=DEVICE):
    """Reuses the same doc-level pipeline as validation (encode -> windows -> CRF decode -> char spans)."""
    model.eval()
    predictions = {}
    for doc_id in tqdm(text_dict, desc="Predicting"):
        text = get_doc_text(text_dict[doc_id])
        if not text.strip():
            predictions[doc_id] = []
            continue
        all_ids, all_char_spans = encode_with_offsets(tokenizer, text)
        n = len(all_ids)
        if n == 0:
            predictions[doc_id] = []
            continue
        step = cfg.window_stride if cfg.window_stride > 0 else cfg.window_size
        starts = list(range(0, max(1, n - cfg.window_size + 1), step))
        if starts[-1] + cfg.window_size < n:
            starts.append(max(0, n - cfg.window_size))
        token_logits = [None] * n
        token_dist   = [-1] * n
        for s in starts:
            e = min(s + cfg.window_size, n)
            w_ids = all_ids[s:e]
            pad_len = cfg.window_size - len(w_ids)
            inp = torch.tensor([w_ids + [0]*pad_len], dtype=torch.long, device=device)
            amask = torch.tensor([[1]*len(w_ids) + [0]*pad_len], dtype=torch.long, device=device)
            with torch.amp.autocast('cuda', enabled=cfg.use_amp):
                out = model(inp, attention_mask=amask)
            logits = out["logits"][0, :len(w_ids), :]
            for i_in_w in range(len(w_ids)):
                g = s + i_in_w
                dist = min(i_in_w, len(w_ids) - 1 - i_in_w)
                if dist > token_dist[g]:
                    token_dist[g] = dist
                    token_logits[g] = logits[i_in_w]
        emissions_list = []
        for i in range(n):
            if token_logits[i] is not None:
                emissions_list.append(token_logits[i])
            else:
                emissions_list.append(torch.zeros(model.num_tags, device=device))
        emissions = torch.stack(emissions_list).unsqueeze(0)
        crf_mask = torch.ones(1, n, dtype=torch.bool, device=device)
        tag_preds = model.crf.decode(emissions, crf_mask)[0]
        trimmed = tag_preds_to_char_spans(tag_preds, all_char_spans, text)
        predictions[doc_id] = trimmed
    return predictions

def save_submission(predictions, exp_name):
    pred_path = SUB_DIR / f"prediction_{exp_name}.json"
    zip_path  = SUB_DIR / f"submission_{exp_name}.zip"
    with open(pred_path, "w") as f:
        json.dump(predictions, f, ensure_ascii=False)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_path, "prediction.json")
    n_ents = sum(len(v) for v in predictions.values())
    print(f"  {exp_name}: {len(predictions)} docs, {n_ents} entities -> {zip_path.name}")
    return zip_path

print("Prediction & submission defined")

Prediction & submission defined


# Run Experiment: exp1_full_ft_bio_fixed

In [31]:
set_seed(CFG.seed)
backbone_1 = load_phase1_backbone(CFG, PHASE1_CKPT, DEVICE)
model_1 = LegalNER(backbone_1, CFG.bio_num_tags, dropout=0.3, freeze_layers=14)
# freeze 14/19 layers — only last 5 layers trainable + BiLSTM + CRF

cfg_1 = copy.copy(CFG)
cfg_1.lr = 5e-6
cfg_1.head_lr = 5e-4
cfg_1.warmup_steps = 200
cfg_1.num_epochs = 6
cfg_1.min_lr = 1e-7
model_1 = train_ner(model_1, train_dl, val_text, val_gold, cfg_1, "exp1_full_ft_bio_fixed", patience=2)

preds_1 = predict_test(model_1, fold3_text, tokenizer, CFG)
save_submission(preds_1, "exp1_full_ft_bio_fixed")

TEACHER_CKPT = CKPT_DIR / "teacher_full_ft.pt"
torch.save(model_1.state_dict(), TEACHER_CKPT)
print("Teacher checkpoint saved locally for Exp 6")

del backbone_1
torch.cuda.empty_cache()

Phase 1 backbone loaded from best.pt
  Parameters: 150,287,616
Frozen 127/173 backbone param tensors (first 14/19 layers + embeddings)
Backbone: 31,465,728 (lr=5.0e-06) | Head: 7,110,076 (lr=5.0e-04)
  [exp1_full_ft_bio_fixed] Ep 1 step 100/1520 | loss 588.3537 | 54s
  [exp1_full_ft_bio_fixed] Ep 1 step 200/1520 | loss 350.9371 | 108s
  [exp1_full_ft_bio_fixed] Ep 1 step 300/1520 | loss 249.0240 | 162s
  [exp1_full_ft_bio_fixed] Ep 1 step 400/1520 | loss 192.3929 | 217s
  [exp1_full_ft_bio_fixed] Ep 1 step 500/1520 | loss 156.4401 | 271s
  [exp1_full_ft_bio_fixed] Ep 1 step 600/1520 | loss 131.8185 | 325s
  [exp1_full_ft_bio_fixed] Ep 1 step 700/1520 | loss 113.8632 | 379s
  [exp1_full_ft_bio_fixed] Ep 1 step 800/1520 | loss 100.2761 | 433s
  [exp1_full_ft_bio_fixed] Ep 1 step 900/1520 | loss 89.5883 | 487s
  [exp1_full_ft_bio_fixed] Ep 1 step 1000/1520 | loss 80.9505 | 541s
  [exp1_full_ft_bio_fixed] Ep 1 step 1100/1520 | loss 73.8826 | 595s
  [exp1_full_ft_bio_fixed] Ep 1 step 1200/1

Predicting:   0%|          | 0/35 [00:00<?, ?it/s]

  exp1_full_ft_bio_fixed: 35 docs, 4178 entities -> submission_exp1_full_ft_bio_fixed.zip
Teacher checkpoint saved locally for Exp 6


In [32]:
# Submission zips are in ./submissions/
for zf in sorted(SUB_DIR.glob("submission_*.zip")):
    print(f"Ready: {zf}")
print("Done! Upload the zip to the leaderboard.")

Ready: submissions/submission_exp1_full_ft_bio_fixed.zip
Done! Upload the zip to the leaderboard.
